In [1]:
import liana
print(liana.__version__)

1.7.3


In [2]:
# ============================================================
# 08_phase3_LIANA_clean.ipynb
# Phase 3 — Cell-Cell Communication Analysis (LIANA)
# Uses corrected cluster annotations from Phase 2
# Supersedes LIANA results in notebook 05
# ============================================================

# ----------------------------
# Cell 1 — Imports and paths
# ----------------------------
import scanpy as sc
import pandas as pd
import numpy as np
import liana as li
import gc
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\annam\Dissertation 2026")
PROCESSED_DIR = PROJECT_DIR / "Data" / "Processed"
RESULTS_DIR = PROJECT_DIR / "results" / "phase3"
FIGURE_DIR = PROJECT_DIR / "figures" / "phase3"

cluster_labels_1 = {
    "0": "T cells",
    "1": "Activated T cells",
    "2": "NK/Cytotoxic T cells",
    "3": "Macrophages",
    "4": "B cells",
    "5": "Monocytes/DC"
}

print("Ready")

Ready


In [3]:
# ----------------------------
# Cell 2 — Load GSE114725 annotated object
# ----------------------------
adata1 = sc.read_h5ad(PROCESSED_DIR / "GSE114725_phase2_v2_annotated.h5ad")
adata1.obs["cell_type"] = adata1.obs["leiden_0.6"].map(cluster_labels_1)

print(f"GSE114725: {adata1.n_obs} cells")
print(adata1.obs["cell_type"].value_counts())
print(f"\nX max (should be log-normalised, ~7-8): {adata1.X.max()}")

GSE114725: 44662 cells
cell_type
T cells                 16554
NK/Cytotoxic T cells     9811
Macrophages              8624
Activated T cells        5353
Monocytes/DC             3530
B cells                   790
Name: count, dtype: int64

X max (should be log-normalised, ~7-8): 10.0


In [4]:
import numpy as np
from scipy.sparse import issparse

X_sample = adata1.X[0:1000]
if issparse(X_sample):
    X_sample = X_sample.toarray()

print(f"Min: {X_sample.min()}")
print(f"Max: {X_sample.max()}")
print(f"Mean: {X_sample.mean():.4f}")
print(f"Are values integers: {np.all(X_sample == X_sample.astype(int))}")

Min: -4.233314997940529
Max: 10.0
Mean: 0.0440
Are values integers: False


In [5]:
# Use raw (log-normalised, unscaled) for LIANA
adata1_liana = adata1.raw.to_adata()
adata1_liana.obs["cell_type"] = adata1.obs["cell_type"].values

X_check = adata1_liana.X[0:1000]
if issparse(X_check): X_check = X_check.toarray()
print(f"Min: {X_check.min()}")
print(f"Max: {X_check.max()}")
print(f"Mean: {X_check.mean():.4f}")

Min: 0.0
Max: 7.098389148712158
Mean: 0.1141


In [6]:
# ----------------------------
# Cell 3 — Run LIANA on GSE114725
# rank_aggregate combines CellPhoneDB, Connectome, NATMI, SingleCellSignalR, log2FC
# ----------------------------
li.mt.rank_aggregate(
    adata1_liana,
    groupby="cell_type",
    expr_prop=0.1,
    verbose=True,
    use_raw=False
)

print("LIANA complete for GSE114725")
print(adata1_liana.uns["liana_res"].head())

Using resource `consensus`.
Using `.X`!
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\anndata\_core\anndata.py:381: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\_pipe_utils\_pre.py:168: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
['TCONS_00029157'] contain `_`. Consider replacing those!
0.33 of entities in the resource are missing from the data.


Generating ligand-receptor stats for 44662 samples and 1163 features


C:\Users\annam\anaconda3\envs\scrna\lib\functools.py:889: UserWarning: zero-centering a sparse array/matrix densifies it.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\sc\_liana_pipe.py:288: ImplicitModificationWarning: Setting element `.layers['scaled']` of view, initializing view as actual.


Assuming that counts were `natural` log-normalized!
Running CellPhoneDB


100%|██████████| 1000/1000 [01:12<00:00, 13.81it/s]


Running Connectome
Running log2FC
Running NATMI
Running SingleCellSignalR
LIANA complete for GSE114725
                    source                target ligand_complex  \
2074  NK/Cytotoxic T cells  NK/Cytotoxic T cells            B2M   
2303               T cells  NK/Cytotoxic T cells            B2M   
990            Macrophages  NK/Cytotoxic T cells            B2M   
300      Activated T cells  NK/Cytotoxic T cells            B2M   
1713          Monocytes/DC  NK/Cytotoxic T cells            B2M   

     receptor_complex  lr_means  cellphone_pvals  expr_prod  scaled_weight  \
2074            KLRD1  2.924317              0.0   4.073163       0.301079   
2303            KLRD1  2.836594              0.0   3.931388       0.163738   
990             KLRD1  2.822353              0.0   3.908373       0.141431   
300             KLRD1  2.821721              0.0   3.907351       0.140463   
1713            KLRD1  2.755051              0.0   3.799602       0.036082   

      lr_logfc  spec_weig

In [8]:
# ----------------------------
# Cell 5 — Load GSE176078 annotated object, restrict to immune cells
# Restricted to immune subset due to memory constraints (as done originally)
# ----------------------------
adata2 = sc.read_h5ad(
    PROCESSED_DIR / "GSE176078_phase2_v2_annotated_corrected.h5ad",
    backed="r"
)

cluster_labels_2 = {
    "0": "Endothelial cells", "1": "Endothelial cells",
    "2": "CAFs", "3": "PVL", "4": "Basal epithelial",
    "5": "B cells", "6": "Cycling cells", "7": "Plasma cells",
    "8": "Cycling epithelial", "9": "CD8 T cells",
    "10": "NK cells", "11": "T cells", "12": "Naive/memory T cells",
    "13": "Luminal epithelial", "14": "Macrophages",
    "15": "Unassigned", "16": "Unassigned", "17": "pDC",
    "18": "Luminal epithelial", "19": "Luminal epithelial",
    "20": "Epithelial (ambiguous)", "21": "Epithelial (ambiguous)",
    "22": "Luminal epithelial", "23": "Luminal epithelial",
    "24": "Luminal epithelial", "25": "Luminal epithelial"
}

adata2.obs["cell_type"] = adata2.obs["leiden_0.6"].map(cluster_labels_2)

# Restrict to immune cell types only
immune_types = ["T cells", "CD8 T cells", "NK cells", "Naive/memory T cells",
                 "B cells", "Plasma cells", "Macrophages", "Monocytes/DC", "pDC"]

immune_mask = adata2.obs["cell_type"].isin(immune_types)
print(f"Immune cells: {immune_mask.sum()} / {adata2.n_obs}")

# Load into memory only the immune subset, using raw (log-normalised)
adata2_immune = adata2[immune_mask].to_memory()
adata2.file.close()
gc.collect()

print(f"\nLoaded: {adata2_immune.n_obs} cells")
print(adata2_immune.obs["cell_type"].value_counts())

Immune cells: 43798 / 91425

Loaded: 43798 cells
cell_type
Naive/memory T cells    11590
CD8 T cells              9387
Macrophages              8705
T cells                  5989
B cells                  2791
Plasma cells             2583
NK cells                 2438
pDC                       315
Name: count, dtype: int64


In [9]:
# Use raw (log-normalised, unscaled) for LIANA
adata2_liana = adata2_immune.raw.to_adata()
adata2_liana.obs["cell_type"] = adata2_immune.obs["cell_type"].values

X_check = adata2_liana.X[0:1000]
from scipy.sparse import issparse
if issparse(X_check): X_check = X_check.toarray()
print(f"Min: {X_check.min()}")
print(f"Max: {X_check.max()}")
print(f"Mean: {X_check.mean():.4f}")

del adata2_immune
gc.collect()

Min: 0.0
Max: 7.366420269012451
Mean: 0.0655


285

In [10]:
# ----------------------------
# Cell 6 — Run LIANA on GSE176078 immune subset
# ----------------------------
li.mt.rank_aggregate(
    adata2_liana,
    groupby="cell_type",
    expr_prop=0.1,
    verbose=True,
    use_raw=False
)

print("LIANA complete for GSE176078")
print(adata2_liana.uns["liana_res"].head())

Using resource `consensus`.
Using `.X`!
Converting to sparse csr matrix!
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\anndata\_core\anndata.py:381: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
2415 features of mat are empty, they will be removed.
Converting `cell_type` to categorical!
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\_pipe_utils\_pre.py:308: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\_pipe_utils\_pre.py:168: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
0.10 of entities in the resource are missing from the data.


Generating ligand-receptor stats for 43798 samples and 1648 features


C:\Users\annam\anaconda3\envs\scrna\lib\functools.py:889: UserWarning: zero-centering a sparse array/matrix densifies it.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\sc\_liana_pipe.py:288: ImplicitModificationWarning: Setting element `.layers['scaled']` of view, initializing view as actual.


Assuming that counts were `natural` log-normalized!
Running CellPhoneDB


100%|██████████| 1000/1000 [01:57<00:00,  8.49it/s]


Running Connectome
Running log2FC
Running NATMI
Running SingleCellSignalR
LIANA complete for GSE176078
                    source    target ligand_complex receptor_complex  \
4462               T cells  NK cells            B2M            KLRD1   
800            CD8 T cells  NK cells            B2M            KLRD1   
3857          Plasma cells  NK cells            B2M            KLRD1   
2791              NK cells  NK cells            B2M            KLRD1   
3339  Naive/memory T cells  NK cells            B2M            KLRD1   

      lr_means  cellphone_pvals  expr_prod  scaled_weight  lr_logfc  \
4462  3.692074              0.0  10.729496       1.361753  1.564621   
800   3.640364              0.0  10.523840       1.252429  1.469877   
3857  3.625818              0.0  10.465988       1.221668  1.489492   
2791  3.622681              0.0  10.453510       1.215029  1.438252   
3339  3.569697              0.0  10.242786       1.102978  1.350717   

      spec_weight   lrscore  specific

In [11]:
old_sig2 = pd.read_csv(RESULTS_DIR / "GSE176078_liana_significant.csv")
new_sig2 = adata2_liana.uns["liana_res"]
new_sig2 = new_sig2[new_sig2["specificity_rank"] < 0.05].copy()

print(f"Old significant interactions: {len(old_sig2)}")
print(f"New significant interactions: {len(new_sig2)}")

print(f"\nOld cell types involved:")
print(sorted(pd.concat([old_sig2["source"], old_sig2["target"]]).unique()))

print(f"\nNew cell types involved:")
print(sorted(pd.concat([new_sig2["source"], new_sig2["target"]]).unique()))

Old significant interactions: 3086
New significant interactions: 413

Old cell types involved:
['B cells', 'CD8 T cells', 'Macrophages', 'Monocytes/DC', 'NK cells', 'Naive/memory T cells', 'Plasma cells', 'T cells', 'pDC']

New cell types involved:
['B cells', 'CD8 T cells', 'Macrophages', 'NK cells', 'Naive/memory T cells', 'Plasma cells', 'T cells', 'pDC']


In [12]:
# Check Monocytes/DC presence in current run
print("Cell type counts going into LIANA:")
print(adata2_immune.obs["cell_type"].value_counts() if 'adata2_immune' in dir() else "adata2_immune not in memory")

# Check expr_prop threshold effect - was it different before?
print(f"\nCurrent run used expr_prop=0.1")

# Compare specificity rank distributions
print(f"\nOld specificity_rank stats:")
print(old_sig2["specificity_rank"].describe())
print(f"\nNew specificity_rank stats:")
print(new_sig2["specificity_rank"].describe())

# Check total interactions tested (not just significant)
print(f"\nTotal interactions tested - new run: {len(liana_res2)}")

Cell type counts going into LIANA:
adata2_immune not in memory

Current run used expr_prop=0.1

Old specificity_rank stats:
count    3.086000e+03
mean     1.730388e-02
std      1.396187e-02
min      9.272740e-07
25%      1.551111e-03
50%      2.147552e-02
75%      2.636458e-02
max      4.989558e-02
Name: specificity_rank, dtype: float64

New specificity_rank stats:
count    413.000000
mean       0.018374
std        0.014745
min        0.000002
25%        0.004780
50%        0.016022
75%        0.030455
max        0.049771
Name: specificity_rank, dtype: float64


NameError: name 'liana_res2' is not defined

In [13]:
# Reload to check why Monocytes/DC is missing
adata2_check = sc.read_h5ad(
    PROCESSED_DIR / "GSE176078_phase2_v2_annotated_corrected.h5ad",
    backed="r"
)
adata2_check.obs["cell_type"] = adata2_check.obs["leiden_0.6"].map(cluster_labels_2)

print("Full GSE176078 cell type counts:")
print(adata2_check.obs["cell_type"].value_counts())

print(f"\nimmune_types list used: {immune_types}")
print(f"Is 'Monocytes/DC' in immune_types list? {'Monocytes/DC' in immune_types}")

adata2_check.file.close()

Full GSE176078 cell type counts:
cell_type
Luminal epithelial        22894
Naive/memory T cells      11590
CD8 T cells                9387
Macrophages                8705
Endothelial cells          7040
CAFs                       6453
T cells                    5989
PVL                        5051
Cycling epithelial         3010
B cells                    2791
Plasma cells               2583
NK cells                   2438
Cycling cells              1159
Basal epithelial           1074
Epithelial (ambiguous)      905
pDC                         315
Unassigned                   41
Name: count, dtype: int64

immune_types list used: ['T cells', 'CD8 T cells', 'NK cells', 'Naive/memory T cells', 'B cells', 'Plasma cells', 'Macrophages', 'Monocytes/DC', 'pDC']
Is 'Monocytes/DC' in immune_types list? True


In [14]:
# Check overlap in actual top interactions, not just counts
old_top = set(zip(
    old_sig2.sort_values("specificity_rank").head(30)["source"],
    old_sig2.sort_values("specificity_rank").head(30)["target"],
    old_sig2.sort_values("specificity_rank").head(30)["ligand_complex"]
))
new_top = set(zip(
    new_sig2.sort_values("specificity_rank").head(30)["source"],
    new_sig2.sort_values("specificity_rank").head(30)["target"],
    new_sig2.sort_values("specificity_rank").head(30)["ligand_complex"]
))

print(f"Overlap in top 30 interactions: {len(old_top & new_top)} / 30")

# Check if CCL19-CCR7 (your headline finding) still appears
print("\nCCL19-CCR7 in old results:")
print(old_sig2[old_sig2["ligand_complex"] == "CCL19"][
    ["source", "target", "specificity_rank"]].to_string())

print("\nCCL19-CCR7 in new results:")
print(new_sig2[new_sig2["ligand_complex"] == "CCL19"][
    ["source", "target", "specificity_rank"]].to_string())

Overlap in top 30 interactions: 3 / 30

CCL19-CCR7 in old results:
            source                target  specificity_rank
3          B cells               B cells          0.034011
105        B cells          Monocytes/DC          0.026365
152        B cells  Naive/memory T cells          0.030643
222        B cells                   pDC          0.026365
578    Macrophages               B cells          0.026365
662    Macrophages           CD8 T cells          0.047921
860    Macrophages          Monocytes/DC          0.026365
1023   Macrophages  Naive/memory T cells          0.026365
1201   Macrophages               T cells          0.041900
1303   Macrophages                   pDC          0.026365
1383  Monocytes/DC               B cells          0.000082
1384  Monocytes/DC               B cells          0.000885
1385  Monocytes/DC               B cells          0.000021
1386  Monocytes/DC               B cells          0.000722
1428  Monocytes/DC           CD8 T cells        

In [15]:
liana_res2.to_csv(RESULTS_DIR / "GSE176078_liana_results.csv", index=False)
liana_sig2.to_csv(RESULTS_DIR / "GSE176078_liana_significant.csv", index=False)

print("New top 15 significant interactions (corrected):")
print(new_sig2.sort_values("specificity_rank").head(15)[
    ["source", "target", "ligand_complex", "receptor_complex", "specificity_rank"]
].to_string())

NameError: name 'liana_res2' is not defined

In [16]:
print("new_sig2 available:", 'new_sig2' in dir())
print("liana_res2 available:", 'liana_res2' in dir())

new_sig2 available: True
liana_res2 available: False


In [17]:
# Reload immune subset and re-run LIANA for GSE176078 (clean save this time)
adata2 = sc.read_h5ad(
    PROCESSED_DIR / "GSE176078_phase2_v2_annotated_corrected.h5ad",
    backed="r"
)
adata2.obs["cell_type"] = adata2.obs["leiden_0.6"].map(cluster_labels_2)

immune_types = ["T cells", "CD8 T cells", "NK cells", "Naive/memory T cells",
                 "B cells", "Plasma cells", "Macrophages", "pDC"]
immune_mask = adata2.obs["cell_type"].isin(immune_types)
adata2_immune = adata2[immune_mask].to_memory()
adata2.file.close()
gc.collect()

adata2_liana = adata2_immune.raw.to_adata()
adata2_liana.obs["cell_type"] = adata2_immune.obs["cell_type"].values
del adata2_immune
gc.collect()

print(f"Ready: {adata2_liana.n_obs} cells")

Ready: 43798 cells


In [18]:
# Run LIANA and save immediately to avoid losing results again
li.mt.rank_aggregate(
    adata2_liana,
    groupby="cell_type",
    expr_prop=0.1,
    verbose=True,
    use_raw=False
)

liana_res2 = adata2_liana.uns["liana_res"]
liana_sig2 = liana_res2[liana_res2["specificity_rank"] < 0.05].copy()

liana_res2.to_csv(RESULTS_DIR / "GSE176078_liana_results.csv", index=False)
liana_sig2.to_csv(RESULTS_DIR / "GSE176078_liana_significant.csv", index=False)

print(f"Saved. Total tested: {len(liana_res2)}, Significant: {len(liana_sig2)}")
print("\nTop 15 significant interactions:")
print(liana_sig2.sort_values("specificity_rank").head(15)[
    ["source", "target", "ligand_complex", "receptor_complex", "specificity_rank"]
].to_string())

Using resource `consensus`.
Using `.X`!
Converting to sparse csr matrix!
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\anndata\_core\anndata.py:381: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
2415 features of mat are empty, they will be removed.
Converting `cell_type` to categorical!
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\_pipe_utils\_pre.py:308: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\_pipe_utils\_pre.py:168: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
0.10 of entities in the resource are missing from the data.


Generating ligand-receptor stats for 43798 samples and 1648 features


C:\Users\annam\anaconda3\envs\scrna\lib\functools.py:889: UserWarning: zero-centering a sparse array/matrix densifies it.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\sc\_liana_pipe.py:288: ImplicitModificationWarning: Setting element `.layers['scaled']` of view, initializing view as actual.


Assuming that counts were `natural` log-normalized!
Running CellPhoneDB


100%|██████████| 1000/1000 [02:22<00:00,  7.00it/s]


Running Connectome
Running log2FC
Running NATMI
Running SingleCellSignalR
Saved. Total tested: 5586, Significant: 413

Top 15 significant interactions:
           source        target ligand_complex receptor_complex  specificity_rank
2057  Macrophages  Plasma cells          CXCL8             SDC1          0.000002
2068  Macrophages  Plasma cells            FN1             SDC1          0.000005
5136          pDC   Macrophages       SERPINF1           PLXDC2          0.000012
1427  Macrophages   Macrophages           C1QB             LRP1          0.000025
1641  Macrophages   Macrophages         S100A9             CD68          0.000025
1423  Macrophages   Macrophages           C1QA             CD93          0.000043
1422  Macrophages   Macrophages           C1QA             CD33          0.000060
1426  Macrophages   Macrophages           C1QB             CD33          0.000062
1652  Macrophages   Macrophages       SERPINA1             LRP1          0.000064
2056  Macrophages  Plasma ce